# テキスト生成アプリを作る

このカリキュラムを通じて、プロンプトなどのコア概念や「プロンプトエンジニアリング」という分野があることを見てきました。ChatGPT、Office 365、Microsoft Power Platformなど、多くのツールがプロンプトを使ってさまざまなことを実行する方法をサポートしています。

こうした体験をアプリに追加するためには、プロンプトや完成形（completions）といった概念を理解し、使うライブラリを選ぶ必要があります。この章ではまさにそれを学びます。

## はじめに

この章では以下を行います:

- openaiライブラリとそのコア概念について学ぶ。
- openaiを使ってテキスト生成アプリを作る。
- プロンプト、temperature、トークンなどの概念を使ってテキスト生成アプリを構築する方法を理解する。

## 学習目標

このレッスンの終わりには、以下ができるようになります:

- テキスト生成アプリとは何か説明できる。
- openaiを使ってテキスト生成アプリを作成できる。
- アプリをトークン数の増減やtemperatureの変更で調整し、多様な結果を出せるようにする。

## テキスト生成アプリとは？

通常アプリを作る場合、以下のようなインターフェースがあります:

- コマンドベース。コンソールアプリは典型的なアプリで、コマンドを打ち込むことで何かを実行します。例えば`git`はコマンドベースのアプリです。
- ユーザーインターフェース（UI）。ボタンをクリックしたり、テキストを入力したり、オプションを選んだりするGUIを持つアプリもあります。

### コンソールとUIアプリの限界

コマンドを入力するコマンドベースのアプリと比較すると: 

- <strong>限界がある</strong>。打てるコマンドはアプリがサポートするものだけです。
- <strong>特定言語向け</strong>。多言語対応のアプリもあるが、基本的に特定言語用に作られており、拡張も可能だが制限があります。 

### テキスト生成アプリの利点

テキスト生成アプリはどう違うでしょう？

より柔軟性があり、設定されたコマンドや特定入力言語に制限されません。自然言語でアプリとやりとりできるのが特徴です。さらに、従来のアプリではデータベースにある内容に限られますが、大量のコーパスでトレーニングされたデータソースとやりとりしています。

### テキスト生成アプリで何が作れる？

例として、以下のようなものが作れます:

- <strong>チャットボット</strong>。会社や製品について質問に答えるチャットボットがよく合います。
- <strong>ヘルパー</strong>。LLMはテキストの要約、洞察抽出、履歴書の作成などに優れています。
- <strong>コードアシスタント</strong>。言語モデルによっては、コードを書く支援をするコードアシスタントが作れます。例えばGitHub CopilotやChatGPTが該当します。

## どう始める？

LLMと統合するには通常次の二つの方法があります:

- APIを使う。ここではプロンプトを含むWebリクエストを作り、生成されたテキストを受け取ります。
- ライブラリを使う。ライブラリはAPI呼び出しをカプセル化し、使いやすくします。

## ライブラリ/SDK

LLMを扱う代表的なライブラリには以下があります:

- **openai**。モデル接続とプロンプト送信が簡単にできます。

より高度なライブラリには以下があります:

- **Langchain**。Pythonに対応している有名なライブラリです。
- **Semantic Kernel**。Microsoftのライブラリで、C#、Python、Javaに対応しています。

## openaiを使った最初のアプリ

まずは最初のアプリを作ってみましょう。必要なライブラリ、準備、などを確認します。

### openaiのインストール

  > [!NOTE] CodespacesやDevcontainer内でこのノートブックを実行する場合、このステップは不要です。


OpenAIやAzure OpenAIとやりとりするライブラリは多く存在し、C#、Python、JavaScript、Javaなど多くの言語で利用可能です。  
今回は`openai`というPythonライブラリを使うので、`pip`でインストールします。

```bash
pip install openai
```

CodespacesやDevcontainer外で実行する場合は、[Python](https://www.python.org/)もマシンにインストールしてください。

### リソースを作成する

まだの方は以下の手順を行う必要があります:

- Azureのアカウントを作成する <https://azure.microsoft.com/free/>。
- Microsoft Foundryの[Azure OpenAIリソース](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)を作成する。リソースの[作成方法](https://learn.microsoft.com/azure/ai-services/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst)はこちら。


### APIキーとエンドポイントの場所を探す

`openai`ライブラリに使うAPIキーを設定するために、Azure OpenAIリソースの「Keys and Endpoint」セクションで「Key 1」をコピーしてください。

  ![AzureポータルのKeys and Endpointリソースブレード](https://learn.microsoft.com/azure/ai-services/openai/media/quickstarts/endpoint.png?WT.mc_id=academic-105485-koreyst)

情報をコピーしたら、次はライブラリに使わせる設定をします。

> [!NOTE]
> APIキーはコードから分離して管理するのが望ましいです。環境変数を使いましょう。
> - .envファイルに環境変数`AZURE_OPENAI_API_KEY`をAPIキーとして設定します。もしこのコースの前の演習を済ませていれば準備完了です。


### Azureの設定方法

Azure OpenAIを使う場合の設定方法です。Responses APIはAzure OpenAIの<strong>v1エンドポイント</strong>で提供されるため、`OpenAI`クライアントを`<your-endpoint>/openai/v1/`に向けます:

```python
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment = os.environ['AZURE_OPENAI_DEPLOYMENT']
```

上記で設定しているのは:

- `api_key`、Azureポータルで取得したAPIキー。
- `base_url`、APIキー横のAzure OpenAIエンドポイントに`/openai/v1/`を追加したもの。v1エンドポイントを使うため、`api_version`は不要です。
- `deployment`、Foundryポータルで作成したモデル展開の名前。

> [!NOTE]
> `os.environ`は環境変数を読むマッピングで、`AZURE_OPENAI_API_KEY`や`AZURE_OPENAI_ENDPOINT`を読むのに使えます。

## テキスト生成

テキストを生成するには`responses.create`メソッドを使います。例を示します:

```python
prompt = "Complete the following: Once upon a time there was a"

response = client.responses.create(model=deployment, input=prompt, store=False)
print(response.output_text)
```

上のコードではレスポンスオブジェクトを作り、使うモデルとプロンプトを渡しています。生成されたテキストをプリントしています。

### チャット形式のレスポンス

Responses APIは単発のテキスト生成だけでなく、多発のチャットボットにも適しています。`input`リストにメッセージを増やして会話を組み立てます:

```python
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment = os.environ['AZURE_OPENAI_DEPLOYMENT']

response = client.responses.create(model=deployment, input=[{"role": "user", "content": "Hello world"}], store=False)
print(response.output_text)
```

この機能については後の章で詳しく説明します。

## 演習 - 最初のテキスト生成アプリ

Azure OpenAIサービスの設定が済んだので、最初のテキスト生成アプリを作ります。以下の手順に従ってください:


1. 仮想環境を作成し、openaiをインストールします：

  > [!NOTE] この手順は、このノートブックをCodespacesまたはDevcontainer内で実行する場合は必要ありません


In [ ]:
# Create virtual environment
! python -m venv venv
# Activate virtual environment
! source venv/bin/activate
# Install openai package
! pip install openai

> [!NOTE]
> Windowsを使用している場合は、`source venv/bin/activate` の代わりに `venv\Scripts\activate` と入力してください。

> [!NOTE]
> https://portal.azure.com/ にアクセスし、`Open AI` を検索して `Open AI resource` を選択し、次に `Keys and Endpoint` を選択して `Key 1` の値をコピーすることで、Azure OpenAIキーを見つけることができます。


1. *app.py* ファイルを作成し、以下のコードを記述します:


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']

# add your completion code
prompt = "Complete the following: Once upon a time there was a"

# make a request using the Responses API
response = client.responses.create(model=deployment, input=prompt, store=False)

# print response
print(response.output_text)



    以下のような出力が表示されるはずです:

    ```output
     very unhappy _____.

    むかしむかし、とても悲しい人魚がいました。
    ```


## 目的に応じたさまざまなタイプのプロンプト

これまでに、プロンプトを使ってテキストを生成する方法を見てきました。実際に動作するプログラムも手に入り、異なる種類のテキストを生成するために修正や変更ができる状態です。

プロンプトはさまざまなタスクに使えます。たとえば：

- <strong>テキストの種類を生成する</strong>。例えば、詩やクイズ用の質問などを生成できます。
- <strong>情報を検索する</strong>。プロンプトを使用して、「ウェブ開発におけるCORSとは何か？」のような情報を調べることができます。
- <strong>コードを生成する</strong>。例えば、メールアドレスの検証に使う正規表現を作成したり、ウェブアプリのようなプログラム全体を生成することもできます。

## より実用的なケース：レシピ生成器

家に材料があって何かを料理したいとします。そのためにはレシピが必要です。レシピを見つける方法としては検索エンジンを使ったり、LLM（大規模言語モデル）を利用したりすることができます。

次のようなプロンプトを書くことができます：

> 「鶏肉、ジャガイモ、ニンジンの材料で作る料理のレシピを5つ見せてください。各レシピごとに使われるすべての材料を一覧にしてください」

上記のプロンプトに対して、次のような回答が得られるかもしれません：

```output
1. Roasted Chicken and Vegetables: 
Ingredients: 
- 4 chicken thighs
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 2 tablespoons olive oil
- 2 cloves garlic, minced
- 1 teaspoon dried thyme
- 1 teaspoon dried oregano
- Salt and pepper, to taste

2. Chicken and Potato Stew: 
Ingredients: 
- 2 tablespoons olive oil
- 1 onion, diced
- 2 cloves garlic, minced
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 teaspoon dried oregano
- 1 teaspoon dried thyme
- 1 cup chicken broth
- Salt and pepper, to taste

3. Chicken and Potato Bake: 
Ingredients: 
- 2 tablespoons olive oil
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 onion, diced
- 2 cloves garlic, minced
- 1 teaspoon dried oregano
- 1 teaspoon dried thyme
- 1 cup chicken broth
- Salt and pepper, to taste

4. Chicken and Potato Soup: 
Ingredients: 
- 2 tablespoons olive oil
- 1 onion, diced
- 2 cloves garlic, minced
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 teaspoon dried oregano
- 1 teaspoon dried thyme
- 4 cups chicken broth
- Salt and pepper, to taste

5. Chicken and Potato Hash: 
Ingredients: 
- 2 tablespoons olive oil
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 onion, diced
- 2 cloves garlic, minced
- 1 teaspoon dried oregano
```

この結果は素晴らしく、何を作るか分かりました。この時点で役立つ改善点としては：

- 嫌いな材料やアレルギーのある材料を除外すること。
- 家に材料が全部ない場合に備えて買い物リストを作成すること。

これらの点について、追加のプロンプトを加えましょう：

> 「ニンニクはアレルギーがあるのでレシピから外し、代わりに他の材料を使ってください。また、鶏肉、ジャガイモ、ニンジンが家に既にあることを考慮して、買い物リストを作成してください」

これで、新しい結果が得られます。すなわち：

```output
1. Roasted Chicken and Vegetables: 
Ingredients: 
- 4 chicken thighs
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 2 tablespoons olive oil
- 1 teaspoon dried thyme
- 1 teaspoon dried oregano
- Salt and pepper, to taste

2. Chicken and Potato Stew: 
Ingredients: 
- 2 tablespoons olive oil
- 1 onion, diced
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 teaspoon dried oregano
- 1 teaspoon dried thyme
- 1 cup chicken broth
- Salt and pepper, to taste

3. Chicken and Potato Bake: 
Ingredients: 
- 2 tablespoons olive oil
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 onion, diced
- 1 teaspoon dried oregano
- 1 teaspoon dried thyme
- 1 cup chicken broth
- Salt and pepper, to taste

4. Chicken and Potato Soup: 
Ingredients: 
- 2 tablespoons olive oil
- 1 onion, diced
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 teaspoon dried oregano
- 1 teaspoon dried thyme
- 4 cups chicken broth
- Salt and pepper, to taste

5. Chicken and Potato Hash: 
Ingredients: 
- 2 tablespoons olive oil
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 onion, diced
- 1 teaspoon dried oregano

Shopping List: 
- Olive oil
- Onion
- Thyme
- Oregano
- Salt
- Pepper
```

これがニンニクの入っていない5つのレシピで、家に材料があることを踏まえた買い物リストも得られました。


## 演習 - レシピジェネレーターを作成する

シナリオを実行したので、実演されたシナリオに合うコードを書いてみましょう。これを行うには、以下の手順に従います：

1. 既存の *app.py* ファイルを出発点として使います
1. `prompt` 変数を探して、そのコードを次のように変更します：


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# load environment variables from .env file
load_dotenv()

client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment = os.environ['AZURE_OPENAI_DEPLOYMENT']

prompt = "Show me 5 recipes for a dish with the following ingredients: chicken, potatoes, and carrots. Per recipe, list all the ingredients used"

# make a request using the Responses API
response = client.responses.create(model=deployment, input=prompt, max_output_tokens=600, store=False)

# print response
print(response.output_text)


もし今コードを実行すると、次のような出力が表示されるはずです:

```output
-Chicken Stew with Potatoes and Carrots: 3 tablespoons oil, 1 onion, chopped, 2 cloves garlic, minced, 1 carrot, peeled and chopped, 1 potato, peeled and chopped, 1 bay leaf, 1 thyme sprig, 1/2 teaspoon salt, 1/4 teaspoon black pepper, 1 1/2 cups chicken broth, 1/2 cup dry white wine, 2 tablespoons chopped fresh parsley, 2 tablespoons unsalted butter, 1 1/2 pounds boneless, skinless chicken thighs, cut into 1-inch pieces
-Oven-Roasted Chicken with Potatoes and Carrots: 3 tablespoons extra-virgin olive oil, 1 tablespoon Dijon mustard, 1 tablespoon chopped fresh rosemary, 1 tablespoon chopped fresh thyme, 4 cloves garlic, minced, 1 1/2 pounds small red potatoes, quartered, 1 1/2 pounds carrots, quartered lengthwise, 1/2 teaspoon salt, 1/4 teaspoon black pepper, 1 (4-pound) whole chicken
-Chicken, Potato, and Carrot Casserole: cooking spray, 1 large onion, chopped, 2 cloves garlic, minced, 1 carrot, peeled and shredded, 1 potato, peeled and shredded, 1/2 teaspoon dried thyme leaves, 1/4 teaspoon salt, 1/4 teaspoon black pepper, 2 cups fat-free, low-sodium chicken broth, 1 cup frozen peas, 1/4 cup all-purpose flour, 1 cup 2% reduced-fat milk, 1/4 cup grated Parmesan cheese

-One Pot Chicken and Potato Dinner: 2 tablespoons olive oil, 1 pound boneless, skinless chicken thighs, cut into 1-inch pieces, 1 large onion, chopped, 3 cloves garlic, minced, 1 carrot, peeled and chopped, 1 potato, peeled and chopped, 1 bay leaf, 1 thyme sprig, 1/2 teaspoon salt, 1/4 teaspoon black pepper, 2 cups chicken broth, 1/2 cup dry white wine

-Chicken, Potato, and Carrot Curry: 1 tablespoon vegetable oil, 1 large onion, chopped, 2 cloves garlic, minced, 1 carrot, peeled and chopped, 1 potato, peeled and chopped, 1 teaspoon ground coriander, 1 teaspoon ground cumin, 1/2 teaspoon ground turmeric, 1/2 teaspoon ground ginger, 1/4 teaspoon cayenne pepper, 2 cups chicken broth, 1/2 cup dry white wine, 1 (15-ounce) can chickpeas, drained and rinsed, 1/2 cup raisins, 1/2 cup chopped fresh cilantro
```

> NOTE, your LLM is nondeterministic, so you might get different results every time you run the program.

よし、それでは改善方法を見てみましょう。改善するためには、コードが柔軟であることが重要です。つまり、材料やレシピの数を改善したり変更したりできるようにしたいのです。


1. 以下のようにコードを変更しましょう:


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# load environment variables from .env file
load_dotenv()

client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment = os.environ['AZURE_OPENAI_DEPLOYMENT']

no_recipes = input("No of recipes (for example, 5: ")

ingredients = input("List of ingredients (for example, chicken, potatoes, and carrots: ")

# interpolate the number of recipes into the prompt an ingredients
prompt = f"Show me {no_recipes} recipes for a dish with the following ingredients: {ingredients}. Per recipe, list all the ingredients used"

# make a request using the Responses API
response = client.responses.create(model=deployment, input=prompt, max_output_tokens=600, store=False)

# print response
print(response.output_text)


-ストロベリーショートケーキ: 牛乳、小麦粉、ベーキングパウダー、砂糖、塩、無塩バター、いちご、生クリーム        
-ストロベリーミルク: 牛乳、いちご、砂糖、バニラエッセンス
```

### Improve by adding filter and shopping list

We now have a working app capable of producing recipes and it's flexible as it relies on inputs from the user, both on the number of recipes but also the ingredients used.

To further improve it, we want to add the following:

- **Filter out ingredients**. We want to be able to filter out ingredients we don't like or are allergic to. To accomplish this change, we can edit our existing prompt and add a filter condition to the end of it like so:

    ```python
    filter = input("Filter (for example, vegetarian, vegan, or gluten-free: ")

    prompt = f"Show me {no_recipes} recipes for a dish with the following ingredients: {ingredients}. Per recipe, list all the ingredients used, no {filter}"
    ```

    Above, we add `{filter}` to the end of the prompt and we also capture the filter value from the user.

    An example input of running the program can now look like so:
    
    ```output    
    No of recipes (for example, 5: 3
    List of ingredients (for example, chicken, potatoes, and carrots: onion,milk
    Filter (for example, vegetarian, vegan, or gluten-free: no milk

    1. French Onion Soup

    Ingredients:
    
    -1 large onion, sliced
    -3 cups beef broth
    -1 cup milk
    -6 slices french bread
    -1/4 cup shredded Parmesan cheese
    -1 tablespoon butter
    -1 teaspoon dried thyme
    -1/4 teaspoon salt
    -1/4 teaspoon black pepper
    
    Instructions:
    
    1. In a large pot, sauté onions in butter until golden brown.
    2. Add beef broth, milk, thyme, salt, and pepper. Bring to a boil.
    3. Reduce heat and simmer for 10 minutes.
    4. Place french bread slices on soup bowls.
    5. Ladle soup over bread.
    6. Sprinkle with Parmesan cheese.
    
    2. Onion and Potato Soup
    
    Ingredients:
    
    -1 large onion, chopped
    -2 cups potatoes, diced
    -3 cups vegetable broth
    -1 cup milk
    -1/4 teaspoon black pepper
    
    Instructions:
    
    1. In a large pot, sauté onions in butter until golden brown.
    2. Add potatoes, vegetable broth, milk, and pepper. Bring to a boil.
    3. Reduce heat and simmer for 10 minutes.
    4. Serve hot.
    
    3. Creamy Onion Soup
    
    Ingredients:
    
    -1 large onion, chopped
    -3 cups vegetable broth
    -1 cup milk
    -1/4 teaspoon black pepper
    -1/4 cup all-purpose flour
    -1/2 cup shredded Parmesan cheese
    
    Instructions:
    
    1. In a large pot, sauté onions in butter until golden brown.
    2. Add vegetable broth, milk, and pepper. Bring to a boil.
    3. Reduce heat and simmer for 10 minutes.
    4. In a small bowl, whisk together flour and Parmesan cheese until smooth.
    5. Add to soup and simmer for an additional 5 minutes, or until soup has thickened.
    ```

    As you can see, any recipes with milk in it has been filtered out. But, if you're lactose intolerant, you might want to filter out recipes with cheese in them as well, so there's a need to be clear.

- **Produce a shopping list**. We want to produce a shopping list, considering what we already have at home.

    For this functionality, we could either try to solve everything in one prompt or we could split it up into two prompts. Let's try the latter approach. Here we're suggesting adding an additional prompt, but for that to work, we need to add the result of the former prompt as context to the latter prompt. 

    Locate the part in the code that prints out the result from the first prompt and add the following code below:
    
    ```python
    old_prompt_result = response.output_text
    prompt = "Produce a shopping list for the generated recipes and please don't include ingredients that I already have."
    
    new_prompt = f"{old_prompt_result} {prompt}"
    messages = [{"role": "user", "content": new_prompt}]
    response = client.responses.create(model=deployment, input=messages, max_output_tokens=1200, store=False)
    
    # print response
    print("Shopping list:")
    print(response.output_text)
    ```

    Note the following:

    - We're constructing a new prompt by adding the result from the first prompt to the new prompt: 
    
        ```python
        new_prompt = f"{old_prompt_result} {prompt}"
        messages = [{"role": "user", "content": new_prompt}]
        ```

    - We make a new request, but also considering the number of tokens we asked for in the first prompt, so this time we say `max_output_tokens` is 1200. 

        ```python
        response = client.responses.create(model=deployment, input=messages, max_output_tokens=1200, store=False)
        ```  

        Taking this code for a spin, we now arrive at the following output:

        ```output
        No of recipes (for example, 5: 2
        List of ingredients (for example, chicken, potatoes, and carrots: apple,flour
        Filter (for example, vegetarian, vegan, or gluten-free: sugar
        Recipes:
         or milk.
        
        -Apple and flour pancakes: 1 cup flour, 1/2 tsp baking powder, 1/2 tsp baking soda, 1/4 tsp salt, 1 tbsp sugar, 1 egg, 1 cup buttermilk or sour milk, 1/4 cup melted butter, 1 Granny Smith apple, peeled and grated
        -Apple fritters: 1-1/2 cups flour, 1 tsp baking powder, 1/4 tsp salt, 1/4 tsp baking soda, 1/4 tsp nutmeg, 1/4 tsp cinnamon, 1/4 tsp allspice, 1/4 cup sugar, 1/4 cup vegetable shortening, 1/4 cup milk, 1 egg, 2 cups shredded, peeled apples
        Shopping list:
         -Flour, baking powder, baking soda, salt, sugar, egg, buttermilk, butter, apple, nutmeg, cinnamon, allspice 
        ```
        
- **A word on token length**. We should consider how many tokens we need to generate the text we want. Tokens cost money, so where possible, we should try to be economical with the number of tokens we use. For example, can we phrase the prompt so that we can use less tokens?

   To change tokens used, you can use the `max_output_tokens` parameter. For example, if you want to use 100 tokens, you would do:

    ```python
    response = client.responses.create(model=deployment, input=messages, max_output_tokens=100, store=False)
    ```

- **Experimenting with temperature**. Temperature is something we haven't mentioned so far but is an important context for how our program performs. The higher the temperature value the more random the output will be. Conversely the lower the temperature value the more predictable the output will be. Consider whether you want variation in your output or not.

   To alter the temperature, you can use the `temperature` parameter. For example, if you want to use a temperature of 0.5, you would do:

    ```python
    response = client.responses.create(model=deployment, input=messages, temperature=0.5, store=False)
    ```

   > Note, the closer to 1.0, the more varied the output.



## Assignment

For this assignment, you can choose what to build.

Here are some suggestions:

- Tweak the recipe generator app to improve it further. Play around with temperature values, and the prompts to see what you can come up with.
- Build a "study buddy". This app should be able to answer questions about a topic for example Python, you could have prompts like "What is a certain topic in Python?", or you could have a prompt that says, show me code for a certain topic etc.
- History bot, make history come alive, instruct the bot to play a certain historical character and ask it questions about its life and times. 

## Solution

### Study buddy

- "You're an expert on the Python language

    Suggest a beginner lesson for Python in the following format:
    
    Format:
    - concepts:
    - brief explanation of the lesson:
    - exercise in code with solutions"

Above is a starter prompt, see how you can use it and tweak it to your liking.

### History bot

Here's some prompts you could be using:

- "You are Abe Lincoln, tell me about yourself in 3 sentences, and respond using grammar and words like Abe would have used"
- "You are Abe Lincoln, respond using grammar and words like Abe would have used:

   Tell me about your greatest accomplishments, in 300 words:"

## Knowledge check

What does the concept temperature do?

1. It controls how random the output is.
1. It controls how big the response is.
1. It controls how many tokens are used.

A: 1

What's a good way to store secrets like API keys?

1. In code.
1. In a file.
1. In environment variables.

A: 3, because environment variables are not stored in code and can be loaded from the code. 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
